In [ ]:
import pandas as pd
train_df = pd.read_csv('/content/train_split.csv')
val_df   = pd.read_csv('/content/val_split.csv')

In [ ]:
train_df.dropna(subset=['question1', 'question2'], inplace= True)
val_df.dropna(subset=['question1', 'question2'], inplace= True)

Build training examples

In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

This list comprehension iterates over a Pandas DataFrame (train_df) and converts each row into an InputExample object

In [ ]:

train_examples = [
    InputExample(texts=[r.question1, r.question2], label=float(r.is_duplicate))
    for r in train_df.itertuples()
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=128)

Set up a validation evaluator

In [ ]:
from sentence_transformers.evaluation import BinaryClassificationEvaluator

val_evaluator = BinaryClassificationEvaluator(
    sentences1=val_df['question1'].tolist(),
    sentences2=val_df['question2'].tolist(),
    labels=val_df['is_duplicate'].tolist(),
)

/tmp/ipykernel_551/163459460.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import BinaryClassificationEvaluator


Load the model and set up the loss

In [ ]:
from sentence_transformers import SentenceTransformer, losses

model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.CosineSimilarityLoss(model)

/tmp/ipykernel_551/4135024227.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Train the model

In [15]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=val_evaluator,
    epochs=1,                      # start with 1 — see note below
    warmup_steps=100,
    evaluation_steps=1000,         # run validation every N steps, watch for overfitting
    output_path='/content/drive/MyDrive/ML/finetuned_sbert',
    save_best_model=True,          # keeps the checkpoint with the BEST val score, not just the last
    use_amp=True,                  # mixed precision training. Requires NVIDIA Ampere GPU
)

Step,Training Loss,Validation Loss,Cosine Accuracy,Cosine Accuracy Threshold,Cosine F1,Cosine F1 Threshold,Cosine Precision,Cosine Recall,Cosine Ap,Cosine Mcc
847,0.099493,No log,0.847567,0.658745,0.802390,0.577949,0.744609,0.869891,0.849939,0.677478


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]